# DuckDB + H3 Geospatial Join

This notebook performs the same ZCTA ↔ NWS Forecast Zone join using **DuckDB's community
H3 extension**. It combines the SQL-native DuckDB workflow from notebook 03 with the
hexagonal grid approach from notebook 04 — all inside a single DuckDB connection.

## How this differs from the other notebooks

| Notebook | Engine | Join mechanism |
|---|---|---|
| 03 — DuckDB Spatial | DuckDB + RTREE | `ST_Intersects()` on projected geometry |
| 04 — H3 Python | h3-py + GeoPandas | Set intersection in Python |
| **05 — DuckDB H3** | **DuckDB + H3 extension** | **H3 cell join in SQL** |

The H3 extension exposes functions like `h3_polygon_wkt_to_cells(wkt, resolution)` directly
in DuckDB SQL. Polyfilling and joining both happen inside the database — no Python loops.

## Key trade-off: approximation

Same as notebook 04: results are approximate because polygons are discretized into hexagons.
Only cells whose centers fall inside a polygon are included (default containment mode).
Resolution 8 (~460 m hex edge) is used here.

## Install
The H3 extension is fetched from the DuckDB community repository at runtime — no extra
`pip install` is needed beyond DuckDB itself.

In [1]:
import duckdb
import time

# =========================
#  CONFIG – EDIT THESE
# =========================
WX_FILE  = "/workspace/data/pa_subsets/pa_z_18mr25.parquet"
ZIP_FILE = "/workspace/data/pa_subsets/pa_zcta_tl_2020_us_zcta520.parquet"
OUTPUT_CSV = "/workspace/data/duckdb_h3_geojoin_results.csv"

# H3 resolution: 0 (coarsest) – 15 (finest)
# Resolution 8 → avg hex area ~0.74 km², edge ~460 m
H3_RESOLUTION = 8

# =========================
#  CONNECT & LOAD EXTENSIONS
# =========================
# Use an in-memory DB so this notebook is self-contained and doesn't
# conflict with the persistent geojoin_duckdb.db from notebook 03.
con = duckdb.connect(":memory:")

# spatial is needed for ST_AsText() to convert geometry → WKT for H3
con.execute("INSTALL spatial; LOAD spatial;")

# H3 community extension — downloaded once, cached in DuckDB's extension dir
con.execute("INSTALL h3 FROM community; LOAD h3;")

print("Extensions loaded: spatial, h3")

Extensions loaded: spatial, h3


In [2]:
# =========================
#  LOAD INPUT TABLES
# =========================
# H3 works in EPSG:4326 (lat/lon) — use the raw geometry column, no projection needed.
con.execute(f"""
    CREATE OR REPLACE TABLE wxareas AS
    SELECT * FROM read_parquet('{WX_FILE}');
""")

con.execute(f"""
    CREATE OR REPLACE TABLE zipcodes AS
    SELECT * FROM read_parquet('{ZIP_FILE}');
""")

wx_count  = con.execute("SELECT COUNT(*) FROM wxareas;").fetchone()[0]
zip_count = con.execute("SELECT COUNT(*) FROM zipcodes;").fetchone()[0]

print(f"wxareas rows:  {wx_count}")
print(f"zipcodes rows: {zip_count}")

wxareas rows:  78
zipcodes rows: 1833


In [3]:
# =========================
#  SQL DEFINITIONS
# =========================
polyfill_wx_sql = f"""
    CREATE OR REPLACE TABLE wx_h3 AS
    SELECT
        NAME,
        CWA,
        UNNEST(h3_polygon_wkt_to_cells(ST_AsText(geometry), {H3_RESOLUTION})) AS cell
    FROM wxareas;
"""

polyfill_zip_sql = f"""
    CREATE OR REPLACE TABLE zip_h3 AS
    SELECT
        ZCTA5CE20,
        GEOID20,
        UNNEST(h3_polygon_wkt_to_cells(ST_AsText(geometry), {H3_RESOLUTION})) AS cell
    FROM zipcodes;
"""

# Join on shared H3 cell ID — no geometry math, just integer equality
join_sql = f"""
    SELECT
        z.ZCTA5CE20        AS zipcode,
        z.GEOID20,
        w.NAME             AS weather_name,
        w.CWA              AS cwa,
        COUNT(*)           AS overlap_cells,
        COUNT(*) * h3_get_hexagon_area_avg({H3_RESOLUTION}, 'm^2')
                           AS area_m2_approx
    FROM zip_h3 z
    JOIN wx_h3  w ON z.cell = w.cell
    GROUP BY z.ZCTA5CE20, z.GEOID20, w.NAME, w.CWA
"""

# =========================
#  POLYFILL (setup — not timed)
#  Equivalent to CREATE INDEX in notebook 03.
# =========================
con.execute(polyfill_wx_sql)
con.execute(polyfill_zip_sql)

# =========================
#  COLD RUN (join only)
# =========================
t0 = time.perf_counter()
_ = con.execute(join_sql).fetchall()
cold_time = time.perf_counter() - t0

print(f"Cold join compute only: {cold_time:.3f} seconds")

Cold join compute only: 0.098 seconds


In [4]:
# =========================
#  WARM RUN
#  wx_h3 and zip_h3 tables are already populated — join only.
# =========================
t0 = time.perf_counter()
_ = con.execute(join_sql).fetchall()
warm_time = time.perf_counter() - t0

print(f"Warm join compute only: {warm_time:.3f} seconds")

successful_joins = con.execute(f"SELECT COUNT(*) FROM ({join_sql})").fetchone()[0]
print(f"Successful joins: {successful_joins}")

Warm join compute only: 0.094 seconds
Successful joins: 2080


In [5]:
# =========================
#  EXPORT
# =========================
print("Writing CSV...")
con.execute(f"""
    COPY ({join_sql})
    TO '{OUTPUT_CSV}'
    (HEADER, DELIMITER ',');
""")
print(f"CSV written to: {OUTPUT_CSV}")

Writing CSV...
CSV written to: /workspace/data/duckdb_h3_geojoin_results.csv


In [6]:
# =========================
#  SUMMARY
# =========================
wx_cell_count  = con.execute("SELECT COUNT(*) FROM wx_h3;").fetchone()[0]
zip_cell_count = con.execute("SELECT COUNT(*) FROM zip_h3;").fetchone()[0]
cell_area_m2   = con.execute(
    f"SELECT h3_get_hexagon_area_avg({H3_RESOLUTION}, 'm^2');"
).fetchone()[0]

print("=" * 50)
print(f"H3 resolution:          {H3_RESOLUTION}")
print(f"Mean cell area:         {cell_area_m2:,.0f} m²")
print(f"Total wx cells:         {wx_cell_count:,}")
print(f"Total zip cells:        {zip_cell_count:,}")
print(f"Join pairs found:       {successful_joins:,}")
print(f"Cold time (join only):  {cold_time:.3f}s")
print(f"Warm time (join only):  {warm_time:.3f}s")
print("=" * 50)
print()
print("Note: H3 results are approximate. Thin polygons or small ZCTAs")
print("may be missed if no hex center falls inside them.")
print("Increase H3_RESOLUTION to improve accuracy at the cost of more cells.")

H3 resolution:          8
Mean cell area:         737,328 m²
Total wx cells:         157,139
Total zip cells:        145,810
Join pairs found:       2,080
Cold time (join only):  0.098s
Warm time (join only):  0.094s

Note: H3 results are approximate. Thin polygons or small ZCTAs
may be missed if no hex center falls inside them.
Increase H3_RESOLUTION to improve accuracy at the cost of more cells.
